In [3]:
import json, pathlib, re
import pandas as pd

MINI = pathlib.Path("/content/cot_sweep_mini")                 # raw responses written by the mini sweep
GOLD_PATH = "/content/GOLD54_escalation_failure_df.jsonl"                       # same gold file as the sweep notebook
assert MINI.exists(), f"{MINI} not found: upload/unzip cot_sweep_mini into /content first"

gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8")]
for i, g in enumerate(gold):                                   # this file has no item_id and calls the escalation field esc_answer_idx
    g.setdefault("item_id", i)
    g.setdefault("esc", g.get("esc_answer_idx"))
by_item = {g["item_id"]: g for g in gold}

MAX_TOKENS = 1500                                              # the budget used in the sweep

raw = []
for f in sorted(MINI.glob("*__*.jsonl")):
    raw += [json.loads(l) for l in open(f, encoding="utf-8")]
print(len(raw), "raw responses from", len(list(MINI.glob('*__*.jsonl'))), "files in", MINI)
chk = pd.DataFrame(raw).groupby(["model", "cond"]).item.agg(n="count", items=lambda s: sorted(s))
print(chk.to_string())

130 raw responses from 26 files in /content/cot_sweep_mini
                              n               items
model                 cond                         
claude_haiku_5_5      cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
deepseek_v4_1_flash   cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
gpt_6_luna            cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
grok_4_7              cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
legacy_claude_3_haiku cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
legacy_deepseek       cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
legacy_gemini_flash   cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
legacy_gpt_4o_mini    cot     5  [1, 7, 17, 40, 47]
                      letter  5  [1, 7, 17, 40, 47]
legac

In [4]:
"""Unified answer parser for the escalation-failure evaluation (v2).

One deterministic, model-agnostic cascade replacing the per-model parsers of the original notebooks.
Every response gets exactly one status: explicit | final_line | option_text | truncated | unparseable.
Only the first three yield a letter; truncated/unparseable are reported separately from clinical failure.
"""
import json
import re

LETTERS = "ABCDE"
L = r"([A-E])"

# Ordered, most specific first.
EXPLICIT_PATTERNS = [
    re.compile(r"\\boxed\{\s*\(?" + L + r"\)?\s*\}", re.I),
    re.compile(r"answer\s+is\s*:?\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:final|correct|best|chosen)\s+(?:answer|choice|option)\s*(?:is)?\s*:?\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:^|\n)\s*\**\s*answer\s*:\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:is|be|choose|select|pick)\s*:?\s*\n+\s*\**\(?" + L + r"\)?[\)\.\*:]", re.I),
    re.compile(r"\*\*\s*\(?" + L + r"\)?\s*[\)\.:]?\s*\*\*"),
    # conclusion sentence naming a lettered option: "...management is D) Thiamine", "best choice is (B)"
    re.compile(r"(?:management|step|choice|option|treatment|answer|intervention)\s+(?:is|would be|should be)\s*:?\s*\**\(?" + L + r"\)\s", re.I),
]
FINAL_LINE = re.compile(r"^\W*\(?" + L + r"\)?\s*(?:[\.\):\-]\s*.*)?\W*$", re.S)


def _last_line(text):
    lines = [x for x in text.strip().splitlines() if x.strip()]
    return lines[-1].strip() if lines else ""


def _final_thought(text):
    """MedGemma thinking models wrap reasoning in <unused94>...<unused95>; answer follows the close."""
    if "<unused95>" in text:
        return text.split("<unused95>", 1)[1], True
    if "<unused94>" in text:
        return text, False  # still inside the thought block when generation ended
    return text, True


def looks_truncated(text, max_tokens=500):
    t = text.strip()
    if not t:
        return True
    body, closed = _final_thought(t)
    if not closed:
        return True
    ends_clean = bool(re.search(r"[\.\!\?\)\*\$\"']\s*$|\b[A-E]\s*$", t))
    # ~4 chars/token; a response near the cap that does not end cleanly was cut off
    return (len(t) > 0.8 * max_tokens * 4 * 0.9) and not ends_clean


NEGATION = re.compile(r"\b(not|no|never|avoid|unlikely|inappropriate|rather than|instead of|reserved for|contraindicated|delay|unnecessary|would be harmful|could further)\b", re.I)


def _option_text_match(tail, options):
    """Return a letter if exactly one option's text appears in `tail` (closing sentences)."""
    if not options:
        return None
    if NEGATION.search(tail):  # a negated mention is not a commitment: leave for semantic adjudication
        return None
    tail_l = tail.lower()
    hits = []
    for k, v in options.items():
        v_l = str(v).lower().strip().rstrip(".")
        if len(v_l) >= 4 and v_l in tail_l:
            hits.append(k)
    return hits[0] if len(hits) == 1 else None


def parse_response(text, options=None, max_tokens=500):
    """-> (letter | None, status)"""
    if text is None or not str(text).strip():
        return None, "unparseable"
    text = str(text)
    body, closed = _final_thought(text)
    scope = body if closed else text

    # 1. explicit statements: take the LAST match so a conclusion beats earlier discussion
    for pat in EXPLICIT_PATTERNS:
        ms = list(pat.finditer(scope))
        if ms:
            return ms[-1].group(1).upper(), "explicit"

    # 2. bare letter / "D) text" on the final line
    m = FINAL_LINE.match(_last_line(scope))
    if m:
        return m.group(1).upper(), "final_line"

    # 3. closing sentences name exactly one option
    sents = [s for s in re.split(r"(?<=[\.\!\?])\s+|\n+", scope.strip()) if s.strip()]
    for k in (1, 2):  # final sentence, else final two; accept only an unambiguous single option
        letter = _option_text_match(" ".join(sents[-k:]), options)
        if letter:
            return letter, "option_text"

    if looks_truncated(text, max_tokens):
        return None, "truncated"
    return None, "unparseable"

In [5]:
rows = []
for r in raw:
    g = by_item[r["item"]]
    if r["finish"] == "error":
        letter, status = None, "api_error"
    else:
        letter, status = parse_response(r["response"], g["options"], MAX_TOKENS)
        if r["finish"] == "length" and letter is None: status = "truncated"
    if letter is None:              outcome = "unresolved"
    elif letter == g["answer_idx"]: outcome = "correct"
    elif letter in g["esc"]:        outcome = "wrong_but_escalates"
    else:                           outcome = "failure"
    rows.append(dict(model=r["model"], cond=r["cond"], item=r["item"], letter=letter, status=status,
                     outcome=outcome, finish=r["finish"], out_tokens=r["out_tokens"],
                     answer=g["answer_idx"], esc="".join(g["esc"]), response=r["response"]))
df = pd.DataFrame(rows)
df["is_legacy"] = df.model.str.startswith("legacy_")
df["cell"] = df.letter.fillna("-") + ":" + df.outcome.map({"correct": "OK", "wrong_but_escalates": "esc", "failure": "FAIL", "unresolved": "??"})
print(len(df), "rows scored | items:", sorted(int(i) for i in df.item.unique()), "| models:", df.model.nunique())

130 rows scored | items: [1, 7, 17, 40, 47] | models: 13


In [6]:
print("== parse status per condition ==")
print(df.groupby(["cond", "status"]).size().unstack(fill_value=0).to_string())

bad = df[df.status.isin(["truncated", "unparseable", "api_error"])]
print("\n== rows needing attention ==")
print(bad[["model", "cond", "item", "status", "finish", "out_tokens"]].to_string(index=False) if len(bad) else "none: every response parsed to a letter")

print("\n== outcome counts per model (n = items per cell) ==")
oc = df.pivot_table(index="model", columns=["cond", "outcome"], values="item", aggfunc="count", fill_value=0)
print(oc.to_string())

print("\n== pooled over all models ==")
pool = df.groupby(["cond", "outcome"]).size().unstack(fill_value=0)
pool["n"] = pool.sum(axis=1)
print(pool.to_string())

== parse status per condition ==
status  api_error  explicit  final_line  truncated  unparseable
cond                                                           
cot             8        50           0          5            2
letter          6         3          53          3            0

== rows needing attention ==
                model   cond  item      status finish  out_tokens
legacy_claude_3_haiku    cot     1   api_error  error         NaN
legacy_claude_3_haiku    cot     7   api_error  error         NaN
legacy_claude_3_haiku    cot    17   api_error  error         NaN
legacy_claude_3_haiku    cot    40   api_error  error         NaN
legacy_claude_3_haiku    cot    47   api_error  error         NaN
legacy_claude_3_haiku letter     1   api_error  error         NaN
legacy_claude_3_haiku letter     7   api_error  error         NaN
legacy_claude_3_haiku letter    17   api_error  error         NaN
legacy_claude_3_haiku letter    40   api_error  error         NaN
legacy_claude_3_haiku

In [7]:
# OK = correct, esc = wrong but still an escalating option, FAIL = escalation failure, ?? = unresolved, - = no letter parsed
print("gold answer / escalation set per item:", {int(i): f"{by_item[i]['answer_idx']}/{''.join(by_item[i]['esc'])}" for i in sorted(int(i) for i in df.item.unique())})
grid = df.pivot_table(index="model", columns=["item", "cond"], values="cell", aggfunc="first")
print(grid.to_string())

gold answer / escalation set per item: {1: 'D/AD', 7: 'D/D', 17: 'E/E', 40: 'C/C', 47: 'D/ABCDE'}
item                     1            7            17              40            47       
cond                    cot letter   cot letter   cot  letter     cot letter    cot letter
model                                                                                     
claude_haiku_5_5       D:OK   D:OK  D:OK   D:OK  E:OK    E:OK    C:OK   C:OK   D:OK   D:OK
deepseek_v4_1_flash    D:OK   D:OK  D:OK   D:OK  E:OK    E:OK    C:OK   C:OK   D:OK   D:OK
gpt_6_luna             D:OK   D:OK  D:OK   D:OK  E:OK    E:OK    C:OK   C:OK   D:OK   D:OK
grok_4_7               D:OK   D:OK  D:OK   D:OK  E:OK    E:OK    C:OK   C:OK   D:OK   D:OK
legacy_claude_3_haiku  -:??   -:??  -:??   -:??  -:??    -:??    -:??   -:??   -:??   -:??
legacy_deepseek        D:OK   D:OK  D:OK   D:OK  E:OK    E:OK    C:OK   C:OK   D:OK  A:esc
legacy_gemini_flash    D:OK   D:OK  D:OK   D:OK  E:OK    E:OK    C:OK   C:OK   D:OK

In [8]:
w = df.pivot_table(index=["model", "item"], columns="cond", values="outcome", aggfunc="first").dropna().reset_index()
w["changed"] = w.letter != w.cot
print("== models whose outcome changed under CoT, per item ==")
print(w.groupby("item").changed.agg(changed="sum", of="count").to_string())
print("\n== transitions letter-only -> CoT ==")
print(w[w.changed].groupby(["letter", "cot"]).size().sort_values(ascending=False).to_string() or "none")

L = df.pivot_table(index=["model", "item"], columns="cond", values="letter", aggfunc="first").dropna(how="all")
print("\nsame letter chosen in both conditions:", int((L["letter"] == L["cot"]).sum()), "of", len(L))

== models whose outcome changed under CoT, per item ==
      changed  of
item             
1           1  13
7           1  13
17          2  13
40          3  13
47          4  13

== transitions letter-only -> CoT ==
letter               cot                
correct              unresolved             6
wrong_but_escalates  correct                2
correct              failure                1
                     wrong_but_escalates    1
failure              correct                1

same letter chosen in both conditions: 45 of 56


In [12]:
tok = df.pivot_table(index="model", columns="cond", values="out_tokens", aggfunc="mean")[["letter", "cot"]]
tok["cot / letter (x)"] = (tok["cot"] / tok["letter"]).round(1)
tok["extra tokens"] = (tok["cot"] - tok["letter"]).round(0)
tok.loc["ALL MODELS"] = [df[df.cond == "letter"].out_tokens.mean(), df[df.cond == "cot"].out_tokens.mean(), None, None]
tok.loc["ALL MODELS", "cot / letter (x)"] = round(tok.loc["ALL MODELS", "cot"] / tok.loc["ALL MODELS", "letter"], 1)
tok.loc["ALL MODELS", "extra tokens"] = round(tok.loc["ALL MODELS", "cot"] - tok.loc["ALL MODELS", "letter"], 0)
tok[["letter", "cot"]] = tok[["letter", "cot"]].round(0)
tok = tok.rename(columns={"letter": "letter-only (mean tokens)", "cot": "CoT (mean tokens)"})

cap = df.assign(hit=df.finish.eq("length")).pivot_table(index="model", columns="cond", values="hit", aggfunc="sum")[["letter", "cot"]]
tok["hit cap: letter / CoT"] = [f"{int(cap.loc[m, 'letter'])} / {int(cap.loc[m, 'cot'])}" if m in cap.index
                                else f"{int(cap['letter'].sum())} / {int(cap['cot'].sum())}" for m in tok.index]
print(tok.to_string())
print(f"\nmax CoT response: {int(df[df.cond == 'cot'].out_tokens.max())} tokens (cap {MAX_TOKENS})")

cond                  letter-only (mean tokens)  CoT (mean tokens)  cot / letter (x)  extra tokens hit cap: letter / CoT
model                                                                                                                   
claude_haiku_5_5                          221.0             1214.0               5.5         992.0                 0 / 0
deepseek_v4_1_flash                       242.0              967.0               4.0         726.0                 0 / 0
gpt_6_luna                                 61.0              417.0               6.8         356.0                 0 / 0
grok_4_7                                 1066.0              751.0               0.7        -314.0                 0 / 0
legacy_deepseek                             2.0              567.0             354.6         566.0                 0 / 0
legacy_gemini_flash                         5.0              865.0             173.1         860.0                 0 / 0
legacy_gpt_4o_mini              

In [13]:
def show_pair(model, item, width=100):
    import textwrap
    g = by_item[item]
    print(f"item {item} | gold {g['answer_idx']} | escalating {''.join(g['esc'])} | model {model}\n")
    print(g["question"][-500:], "\n")
    for cond in ("letter", "cot"):
        r = df[(df.model == model) & (df.item == item) & (df.cond == cond)].iloc[0]
        print(f"--- {cond}: parsed {r.letter} -> {r.outcome} ({r.status}, {r.out_tokens} tokens) ---")
        for para in str(r.response).split("\n"):
            print(textwrap.fill(para, width) if para.strip() else "")
        print()

# change the model and item below (use a key and an item number printed by A1)
first_change = w[w.changed].iloc[0] if w.changed.any() else w.iloc[0]
show_pair(first_change.model, int(first_change["item"]))

item 47 | gold D | escalating ABCDE | model legacy_deepseek

ptoms, and the patient calls an ambulance. Emergency medical service arrives within 10 minutes and begins evaluation and prehospital management. The vital signs include: blood pressure 85/50 mm Hg, heart rate 96/min, respiratory rate 19/min, temperature 37.1℃ (98.9℉), and SpO2 89% on ambient air. Oxygen supply and intravenous access are established. An ECG shows the findings in the given image. Which of the following is a part of a proper further prehospital management strategy for this patient? 

--- letter: parsed A -> wrong_but_escalates (final_line, 2.0 tokens) ---
A

--- cot: parsed D -> correct (explicit, 678.0 tokens) ---
Let's analyze the case step by step:

Key findings:
- 57-year-old man with known angina, now experiencing severe retrosternal pain radiating to left
hand (suggestive of acute coronary syndrome, possibly MI).
- Sublingual nitroglycerin x2 (5 min apart) did not improve symptoms – indicates failure of fi

In [17]:
import textwrap

# one row per (model, item) with both conditions side by side
P = df.pivot_table(index=["model", "item"], columns="cond", values=["letter", "outcome", "out_tokens"], aggfunc="first")
P.columns = [f"{a}_{b}" if b else a for a, b in P.columns]
P = P.reset_index().dropna(subset=["outcome_cot", "outcome_letter"])
P["letter_changed"] = P.letter_cot.fillna("-") != P.letter_letter.fillna("-")
P["outcome_changed"] = P.outcome_cot != P.outcome_letter
D = P[P.letter_changed | P.outcome_changed].copy()
D["transition"] = D.outcome_letter + "  ->  " + D.outcome_cot
D["gold"] = D["item"].map(lambda i: by_item[i]["answer_idx"])
D["esc"] = D["item"].map(lambda i: "".join(by_item[i]["esc"]))
print(f"{len(D)} of {len(P)} (model, item) pairs differ between letter-only and CoT "
      f"({int(D.outcome_changed.sum())} change outcome, {int((D.letter_changed & ~D.outcome_changed).sum())} change letter only)\n")
print(D.sort_values(["transition", "model", "item"])[["model", "item", "gold", "esc", "letter_letter", "letter_cot", "transition"]]
      .rename(columns={"letter_letter": "letter-only picks", "letter_cot": "CoT picks"}).to_string(index=False))


def show_diffs(tail=700, width=100, models=None, transitions=None, items=None, full=False, limit=None):
    """Print every differing pair. tail = last N characters of each response (full=True prints everything).
    Filter with models=[...], items=[...], transitions=['correct  ->  failure', ...]; limit=N shows the first N."""
    d = D.sort_values(["transition", "model", "item"])
    if models:      d = d[d.model.isin(models)]
    if items:       d = d[d["item"].isin(items)]
    if transitions: d = d[d.transition.isin(transitions)]
    if limit: d = d.head(limit)
    for n, (_, r) in enumerate(d.iterrows(), 1):
        g = by_item[r["item"]]
        print("=" * width)
        print(f"[{n}/{len(d)}] {r.model} | item {r['item']} | gold {g['answer_idx']} | escalating {''.join(g['esc'])} | {r.transition}")
        print(f"question (end): ...{g['question'][-250:]}")
        for cond, lab in (("letter", "LETTER-ONLY"), ("cot", "CoT")):
            x = df[(df.model == r.model) & (df["item"] == r["item"]) & (df.cond == cond)].iloc[0]
            txt = str(x.response).strip()
            cut = txt if (full or len(txt) <= tail) else "[...] " + txt[-tail:]
            print(f"\n--- {lab}: picked {x.letter} -> {x.outcome} | {x.status} | {x.out_tokens} tokens ---")
            for para in cut.split("\n"):
                print(textwrap.fill(para, width) if para.strip() else "")
        print()

# start with a few, then widen, e.g.:
#   show_diffs(tail=1500, transitions=["correct  ->  failure"])
#   show_diffs(full=True, models=["grok_4_7"])
#   show_diffs(tail=400)   # every differing pair
show_diffs(full=True)

11 of 65 (model, item) pairs differ between letter-only and CoT (11 change outcome, 0 change letter only)

               model  item gold   esc letter-only picks CoT picks                       transition
  legacy_gpt_4o_mini    40    C     C                 C         D             correct  ->  failure
  legacy_gpt_4o_mini    47    D ABCDE                 D      None          correct  ->  unresolved
legacy_mistral_small     7    D     D                 D      None          correct  ->  unresolved
legacy_mistral_small    17    E     E                 E      None          correct  ->  unresolved
legacy_mistral_small    40    C     C                 C      None          correct  ->  unresolved
     mistral_large_4     1    D    AD                 D      None          correct  ->  unresolved
     mistral_large_4    40    C     C                 C      None          correct  ->  unresolved
      legacy_qwen_7b    47    D ABCDE                 D         B correct  ->  wrong_but_escalates
  

In [20]:
import textwrap

if "df_raw" not in globals():
    df_raw = df.copy()                       # untouched copy of the A2 result: overrides are always applied to this

sus = df_raw[(~df_raw.status.isin(["explicit", "final_line"])) | (df_raw.finish != "stop") | (df_raw.letter.isna())].copy()
sus["why"] = sus.apply(lambda r: "api_error" if r.status == "api_error" else "no letter" if pd.isna(r.letter)
                       else "option_text guess" if r.status == "option_text" else "cap hit" if r.finish == "length" else r.status, axis=1)
print(len(sus), "rows to review")
print(sus.groupby(["why", "cond"]).size().unstack(fill_value=0).to_string())
print()
for _, r in sus.sort_values(["why", "model", "cond", "item"]).iterrows():
    g = by_item[r["item"]]
    tail = str(r.response).strip().replace("\n", " ")[-260:]
    print(f"({r.model!r}, {r.cond!r}, {r['item']})  gold {g['answer_idx']} esc {''.join(g['esc'])} | parsed {r.letter} -> {r.outcome} | {r.why} | {r.out_tokens} tok")
    print(textwrap.fill("   ..." + tail, 110, subsequent_indent="   "), "\n")

24 rows to review
cond       cot  letter
why                   
api_error    8       6
no letter    7       3

('legacy_claude_3_haiku', 'cot', 1)  gold D esc AD | parsed None -> unresolved | api_error | nan tok
   ...ERROR: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions 

('legacy_claude_3_haiku', 'cot', 7)  gold D esc D | parsed None -> unresolved | api_error | nan tok
   ...ERROR: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions 

('legacy_claude_3_haiku', 'cot', 17)  gold E esc E | parsed None -> unresolved | api_error | nan tok
   ...ERROR: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions 

('legacy_claude_3_haiku', 'cot', 40)  gold C esc C | parsed None -> unresolved | api_error | nan tok
   ...ERROR: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions 

('legacy_claude_3_haiku', 'cot', 47)  gold D esc ABCDE | parsed None -> unresolved | api_error

In [21]:
OVERRIDES = [
    ("legacy_gpt_4o_mini", "cot", 47, "D", "response ends 'Final answer: Answer: D'; parser missed the doubled 'Answer:'"),
]


def apply_overrides(base, overrides):
    d = base.copy(); d["override_note"] = ""
    for model, cond, item, new, note in overrides:
        m = (d.model == model) & (d.cond == cond) & (d["item"] == item)
        assert m.sum() == 1, f"override {(model, cond, item)} matches {int(m.sum())} rows: check the triple"
        if new == "EXCLUDE":
            d = d[~m]; continue
        g = by_item[item]
        d.loc[m, "letter"] = new
        d.loc[m, "status"] = "manual"
        d.loc[m, "outcome"] = ("unresolved" if new is None else "correct" if new == g["answer_idx"]
                               else "wrong_but_escalates" if new in g["esc"] else "failure")
        d.loc[m, "override_note"] = note
    d["cell"] = d.letter.fillna("-") + ":" + d.outcome.map({"correct": "OK", "wrong_but_escalates": "esc", "failure": "FAIL", "unresolved": "??"})
    return d

df = apply_overrides(df_raw, OVERRIDES)     # every cell from A3 onwards now uses this updated df
json.dump([list(o) for o in OVERRIDES], open("/content/mini_overrides.json", "w"), indent=1)

chg = df_raw.merge(df[["model", "cond", "item", "letter", "outcome"]], on=["model", "cond", "item"], how="left", suffixes=("_before", "_after"))
chg = chg[(chg.letter_before.fillna("-") != chg.letter_after.fillna("-")) | (chg.outcome_before != chg.outcome_after)]
print(f"{len(OVERRIDES)} overrides listed | {len(df_raw) - len(df)} rows excluded | {len(chg[chg.outcome_after.notna()])} rows changed")
print(chg[["model", "cond", "item", "letter_before", "outcome_before", "letter_after", "outcome_after"]].fillna({"outcome_after": "EXCLUDED"}).to_string(index=False) if len(chg) else "")
print("saved the override list to /content/mini_overrides.json (keep it with the results)")

1 overrides listed | 0 rows excluded | 1 rows changed
             model cond  item letter_before outcome_before letter_after outcome_after
legacy_gpt_4o_mini  cot    47          None     unresolved            D       correct
saved the override list to /content/mini_overrides.json (keep it with the results)


In [22]:
w = df.pivot_table(index=["model", "item"], columns="cond", values="outcome", aggfunc="first").dropna().reset_index()
w["changed"] = w.letter != w.cot
print("== models whose outcome changed under CoT, per item ==")
print(w.groupby("item").changed.agg(changed="sum", of="count").to_string())
print("\n== transitions letter-only -> CoT ==")
print(w[w.changed].groupby(["letter", "cot"]).size().sort_values(ascending=False).to_string() or "none")

L = df.pivot_table(index=["model", "item"], columns="cond", values="letter", aggfunc="first").dropna(how="all")
print("\nsame letter chosen in both conditions:", int((L["letter"] == L["cot"]).sum()), "of", len(L))

== models whose outcome changed under CoT, per item ==
      changed  of
item             
1           1  13
7           1  13
17          2  13
40          3  13
47          3  13

== transitions letter-only -> CoT ==
letter               cot                
correct              unresolved             5
wrong_but_escalates  correct                2
correct              failure                1
                     wrong_but_escalates    1
failure              correct                1

same letter chosen in both conditions: 46 of 56


In [23]:
w = df.pivot_table(index=["model", "item"], columns="cond", values="outcome", aggfunc="first").dropna().reset_index()
st = df.pivot_table(index=["model", "item"], columns="cond", values="status", aggfunc="first")
w = w.merge(st.add_prefix("status_").reset_index(), on=["model", "item"])
w["changed"] = w.letter != w.cot
print("== models whose outcome changed under CoT, per item ==")
print(w.groupby("item").changed.agg(changed="sum", of="count").to_string())
print("\n== transitions letter-only -> CoT ==")
print(w[w.changed].groupby(["letter", "cot"]).size().sort_values(ascending=False).to_string() or "none")

# side remark: why did rows become unresolved? (api_error = infrastructure, truncated = token cap, unparseable = parser)
to_unres = w[w.changed & (w.cot == "unresolved")]
if len(to_unres):
    print(f"\nnote: {len(to_unres)} of the changes are '-> unresolved' under CoT. The model gave no usable answer, so they are not clinical choices:")
    why = to_unres.groupby(["letter", "status_cot"]).size().rename("rows").reset_index()
    why["reason"] = why.status_cot.map({"api_error": "API error (e.g. 429 rate limit), re-run these",
                                        "truncated": "hit the token cap before answering",
                                        "unparseable": "parser found no answer, check with O1/peek"})
    print(why.rename(columns={"letter": "letter-only outcome", "status_cot": "CoT status"}).to_string(index=False))
    api = to_unres[to_unres.status_cot == "api_error"]
    if len(api): print("\nAPI-error rows to re-run:", [(r.model, r["item"]) for _, r in api.iterrows()])

L = df.pivot_table(index=["model", "item"], columns="cond", values="letter", aggfunc="first").dropna(how="all")
print("\nsame letter chosen in both conditions:", int((L["letter"] == L["cot"]).sum()), "of", len(L))

== models whose outcome changed under CoT, per item ==
      changed  of
item             
1           1  13
7           1  13
17          2  13
40          3  13
47          3  13

== transitions letter-only -> CoT ==
letter               cot                
correct              unresolved             5
wrong_but_escalates  correct                2
correct              failure                1
                     wrong_but_escalates    1
failure              correct                1

note: 5 of the changes are '-> unresolved' under CoT. The model gave no usable answer, so they are not clinical choices:
letter-only outcome CoT status  rows                                        reason
            correct  api_error     3 API error (e.g. 429 rate limit), re-run these
            correct  truncated     2            hit the token cap before answering

API-error rows to re-run: [('legacy_mistral_small', 7), ('legacy_mistral_small', 17), ('legacy_mistral_small', 40)]

same letter chosen in